# Autiz v2 — two-stage training and evaluation (single Kaggle run)

**What this notebook does.** Stage 1 trains a content scorer on ChaLearn First Impressions V2 transcripts (frozen RoBERTa-large features -> `ContentBranch` -> **C** -> `content_head`). Stage 2 trains a prosody branch **D** to reconstruct 62 z-scored GeMAPS voice features while being statistically independent of the frozen C. It then tests that independence, compares against a prosody-only baseline on 500 synthetic pairs, and evaluates on 355 real clips from 8 speakers with leave-one-speaker-out folds.

**Run order.** Run top to bottom once on a Kaggle GPU with Internet on and both private datasets attached (`autiz-chalearn-fi-v2`, `autiz-real-asd-clips`). Section A starts with a preflight and ends with a self-test on random data; either one stops the run early if anything is wrong.

**Smoke mode.** First set `SMOKE = True` in the config cell for a cheap check that every cell works: 20 pairs, about 300 ChaLearn clips from at least 100 videos, 3 real clips per speaker, 2 epochs, 2 lambda values, 5 permutations, all 8 LOSO folds, real Whisper and RoBERTa. It writes only to `/kaggle/working/smoke_outputs/` and stamps every JSON, CSV and PNG "SMOKE RUN, NOT RESULTS". Its numbers mean nothing. Set `SMOKE = False` (the default) for the real run; the preflight prints `MODE: SMOKE (not results)` or `MODE: FULL` as its first line.

**Honest caveat.** The content score reads only the transcript, so it cannot change when delivery changes. That is true by construction, not a training result. The informative results are the independence tests (section G), reconstruction quality, and the real-speaker evaluation (section J).

**Outputs** (all in `/kaggle/working`, copied into the repo in Phase 2):

| Kaggle file | Repo path |
|---|---|
| `autiz_v2.pt` | `models/checkpoints/autiz_v2.pt` |
| `gemaps_scaler.json` | `models/checkpoints/gemaps_scaler.json` |
| `prosody_baseline.joblib` | `models/checkpoints/prosody_baseline.joblib` |
| `eval_results_synthetic.json`, `eval_results_independence.json`, `eval_results_loso.json`, `eval_results_content.json` | `src/eval/results/` |
| `training_log_v2.csv` | `src/eval/results/training_log_v2.csv` |
| `figures/*.png` | `docs/figures/` |
| `transcripts_real.csv` | **do not commit** (speech content of real speakers) |
| `gemaps_cache.npz`, `cls_*.npy` | caches only |

## A. Setup, preflight, shared code and self-test

In [ ]:
# Install the packages Kaggle does not ship with.
!pip install -q opensmile praat-parselmouth faster-whisper shap

# Patch av.open to ignore metadata_errors parameter (Kaggle's av doesn't support it)
import av
_original_open = av.open

def patched_open(file, mode=None, format=None, options=None, metadata_errors=None, **kwargs):
    "Patched av.open that ignores metadata_errors for compatibility."
    return _original_open(file, mode=mode, format=format, options=options, **kwargs)

av.open = patched_open
print("Patched av.open to ignore metadata_errors parameter")

In [ ]:
# Verify av version supports metadata_errors parameter.
import av
import io

av_version = av.__version__
print(f"av version: {av_version}")

# Test if metadata_errors parameter works by trying to use it
test_data = b"dummy"
test_file = io.BytesIO(test_data)

try:
    container = av.open(test_file, mode="r", metadata_errors="ignore")
    container.close()
    print("SUCCESS: av.open() supports metadata_errors parameter")
except TypeError as e:
    if "metadata_errors" in str(e):
        raise RuntimeError(f"av {av_version} does not support metadata_errors. Need >= 11.0.0. Restart kernel and re-run.")
    raise
except Exception as e:
    # Expected - test file is dummy. If we got past metadata_errors check, OK.
    if "metadata_errors" not in str(e):
        print(f"av version {av_version} is compatible (parameter check passed)")
    else:
        raise RuntimeError(f"av {av_version} does not support metadata_errors. Need >= 11.0.0. Restart kernel and re-run.")

In [ ]:
# Configuration: seeds, run switches and constants used by every later cell.
import os, json, math, time, random, shutil, subprocess, hashlib, datetime, tempfile
import numpy as np
import pandas as pd

SMOKE = False

SEED = 42
N_PAIRS = 500
LOSO_RETRAIN = True
MIN_CLIPS_FOR_INDEPENDENCE = 10
CONTENT_LABEL_SOURCE = "chalearn"
INDEPENDENCE_PENALTY = "xcorr"
LAMBDA_GRID = [0.0, 0.1, 1.0, 10.0, 100.0]
N_PERMUTATIONS = 200
Z_CLIP = 10.0
STD_FLOOR_THRESHOLD = 1e-8
STD_FLOOR_VALUE = 1.0
RUN_DATE = datetime.date.today().isoformat()
PROXY_STAMP = "PROXY, not a trained scorer"
BASELINE_LABEL = "illustrative comparison model, not an evaluator or vendor tool"

DATASET_CHALEARN = "autiz-chalearn-fi-v2"
DATASET_REAL = "autiz-real-asd-clips"
EXPECTED_CHALEARN_CLIPS = 6000
EXPECTED_REAL_CLIPS = {"S001": 110, "S002": 22, "S003": 39, "S004": 35, "S005": 15, "S006": 21, "S007": 3, "S008": 110}
SPEAKER_FOLDER_MAP = {f"speaker_{i:02d}": f"S{i:03d}" for i in range(1, 9)}

ROBERTA_NAME = "roberta-large"
WHISPER_MODEL_NAME = "base"
MAX_PAIR_SECONDS = 10.0
HELD_OUT_FRACTION = 0.2
VAL_FRACTION_OF_TRAIN = 0.2
AROUSAL_FEATURES = [
    "F0semitoneFrom27.5Hz_sma3nz_stddevNorm",
    "loudness_sma3_amean",
    "VoicedSegmentsPerSec",
]
CONTENT_HP = dict(lr=1e-3, weight_decay=1e-2, batch_size=128, max_epochs=200, patience=20)
PROSODY_HP = dict(lr=1e-3, weight_decay=1e-4, batch_size=128, max_epochs=200, patience=20, min_epochs_loso=10)

EXPECTED_N_PAIRS = 500
MIN_TEST_SPEAKERS, MIN_TEST_PAIRS, MIN_BATCH_ROWS = 5, 30, 32
SMOKE_CHALEARN_CLIPS, SMOKE_CHALEARN_MIN_VIDEOS, SMOKE_REAL_PER_SPEAKER = 300, 100, 3
RUN_STAMP = None
if SMOKE:
    N_PAIRS = EXPECTED_N_PAIRS = 20
    LAMBDA_GRID = [0.0, 10.0]
    N_PERMUTATIONS = 5
    CONTENT_HP["max_epochs"] = PROSODY_HP["max_epochs"] = PROSODY_HP["min_epochs_loso"] = 2
    MIN_TEST_SPEAKERS, MIN_TEST_PAIRS, MIN_BATCH_ROWS = 2, 4, 8
    RUN_STAMP = "SMOKE RUN, NOT RESULTS"

WORK = "/kaggle/working"
OUT_DIR = f"{WORK}/smoke_outputs" if SMOKE else WORK
PAIRS_DIR = f"{OUT_DIR}/synthetic_pairs"
REAL_WAV_DIR = f"{OUT_DIR}/real_wav_16k"
FIG_DIR = f"{OUT_DIR}/figures"
OUT = {
    "checkpoint": f"{OUT_DIR}/autiz_v2.pt",
    "scaler": f"{OUT_DIR}/gemaps_scaler.json",
    "baseline": f"{OUT_DIR}/prosody_baseline.joblib",
    "log": f"{OUT_DIR}/training_log_v2.csv",
    "content": f"{OUT_DIR}/eval_results_content.json",
    "independence": f"{OUT_DIR}/eval_results_independence.json",
    "synthetic": f"{OUT_DIR}/eval_results_synthetic.json",
    "loso": f"{OUT_DIR}/eval_results_loso.json",
    "transcripts": f"{OUT_DIR}/transcripts_real.csv",
    "gemaps_cache": f"{OUT_DIR}/gemaps_cache.npz",
}
os.makedirs(FIG_DIR, exist_ok=True)
log_rows = []

T0 = time.time()
_section_clock = {}


def section_start(name):
    """Print a progress line with the minutes elapsed since this config cell ran."""
    _section_clock[name] = time.time()
    print(f"[{(time.time() - T0) / 60:6.1f} min] >>> START {name}")


def section_end(name):
    """Print the elapsed minutes and how long the section took."""
    now = time.time()
    print(f"[{(now - T0) / 60:6.1f} min] <<< END   {name} (took {(now - _section_clock[name]) / 60:.1f} min)")
section_start("A. Setup, preflight, shared code and self-test")

In [ ]:
# Helpers: dataset discovery, ChaLearn loading and splitting, real-clip listing, ffmpeg lookup.
import os
import pickle
import shutil
import subprocess
import sys
import numpy as np
from sklearn.model_selection import GroupShuffleSplit


def find_dataset(name, root="/kaggle/input", max_depth=4):
    """Return the folder called `name` somewhere under `root` (Kaggle mount paths vary); list what exists if absent."""
    seen = []
    for dirpath, dirnames, _ in os.walk(root):
        rel = os.path.relpath(dirpath, root)
        depth = 0 if rel == "." else rel.count(os.sep) + 1
        if depth <= 2:
            seen.append(dirpath)
        if depth > 0 and os.path.basename(dirpath) == name:
            return dirpath
        if depth >= max_depth:
            dirnames[:] = []
    raise FileNotFoundError(f"Dataset folder '{name}' not found under {root}. Folders found: {seen}")


def load_pickle(path):
    """Load a pickle; if it was written by Python 2 (UnicodeDecodeError), retry with encoding latin1."""
    with open(path, "rb") as f:
        try:
            return pickle.load(f)
        except UnicodeDecodeError:
            f.seek(0)
            return pickle.load(f, encoding="latin1")


def video_id_of(filename):
    """YouTube video id: the part of the clip filename before the first dot."""
    return os.path.basename(filename).split(".")[0]


def load_chalearn_training(base, expected_clips):
    """Load the ChaLearn training annotations (interview label only) and transcripts; fail loudly on any mismatch.

    With expected_clips=None (smoke mode) the filenames must still overlap completely, but the total is not fixed."""
    ann_path = os.path.join(base, "train-annotation", "annotation_training.pkl")
    trans_path = os.path.join(base, "train-transcription", "transcription_training.pkl")
    for p in (ann_path, trans_path):
        if not os.path.isfile(p):
            raise FileNotFoundError(f"Missing ChaLearn file: {p}")
    ann = load_pickle(ann_path)
    trans = load_pickle(trans_path)
    if "interview" not in ann:
        raise KeyError(f"'interview' not among annotation keys {sorted(ann)}")
    interview = ann["interview"]
    common = sorted(set(interview) & set(trans))
    if expected_clips is None:
        if not (len(interview) == len(trans) == len(common)):
            raise ValueError(f"ChaLearn mismatch: {len(interview)} annotations, {len(trans)} transcripts, {len(common)} filenames in both.")
    elif not (len(interview) == len(trans) == len(common) == expected_clips):
        raise ValueError(
            f"ChaLearn mismatch: {len(interview)} annotations, {len(trans)} transcripts, "
            f"{len(common)} filenames in both; expected {expected_clips} of each."
        )
    texts = [trans[n] for n in common]
    if not all(isinstance(t, str) for t in texts):
        raise TypeError("Some ChaLearn transcripts are not strings.")
    labels = np.array([float(interview[n]) for n in common], dtype=np.float64)
    if not (np.isfinite(labels).all() and labels.min() >= 0.0 and labels.max() <= 1.0):
        raise ValueError("ChaLearn interview labels are not all finite values in [0, 1].")
    return {
        "names": common,
        "texts": texts,
        "labels": labels,
        "video_ids": np.array([video_id_of(n) for n in common]),
        "n_annotation": len(interview),
        "n_transcription": len(trans),
        "n_common": len(common),
    }


def subset_chalearn(ch, n_clips, min_videos, seed, max_per_video=3):
    """Smoke mode: keep up to `max_per_video` clips from randomly chosen videos until there are about n_clips clips and at least min_videos videos."""
    by_video = {}
    for i, v in enumerate(ch["video_ids"]):
        by_video.setdefault(v, []).append(i)
    videos = sorted(by_video)
    order = [videos[i] for i in np.random.default_rng(seed).permutation(len(videos))]
    keep, n_videos = [], 0
    for v in order:
        keep.extend(by_video[v][:max_per_video])
        n_videos += 1
        if len(keep) >= n_clips and n_videos >= min_videos:
            break
    keep = np.array(sorted(keep))
    out = dict(ch)
    out["names"] = [ch["names"][i] for i in keep]
    out["texts"] = [ch["texts"][i] for i in keep]
    out["labels"] = ch["labels"][keep]
    out["video_ids"] = ch["video_ids"][keep]
    out["n_common"] = len(keep)
    return out


def group_split(groups, test_size, seed):
    """Split row indices in two so that no group id (video id, speaker id) appears on both sides."""
    groups = np.asarray(groups)
    a, b = next(GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=seed).split(np.zeros(len(groups)), groups=groups))
    overlap = set(groups[a]) & set(groups[b])
    assert not overlap, f"{len(overlap)} group ids appear on both sides of the split"
    return a, b


def list_real_clips(wav_root, speaker_map, expected_counts=None, max_per_speaker=None):
    """List the real .wav clips per speaker with anonymised clip ids; fail loudly if folders are missing, empty or miscounted.

    `max_per_speaker` (smoke mode) keeps only the first N clips of each speaker in sorted-filename order."""
    if not os.path.isdir(wav_root):
        raise FileNotFoundError(f"Real-clip folder not found: {wav_root}")
    clips, counts, problems = [], {}, []
    for speaker_id, folder in speaker_map.items():
        d = os.path.join(wav_root, folder)
        if not os.path.isdir(d):
            problems.append(f"missing folder {d}")
            continue
        files = sorted(f for f in os.listdir(d) if f.lower().endswith(".wav"))
        if not files:
            problems.append(f"no .wav files in {d}")
            continue
        counts[folder] = len(files)
        for i, f in enumerate(files[:max_per_speaker]):
            clips.append({"speaker_id": speaker_id, "folder": folder, "src": os.path.join(d, f), "clip_id": f"{speaker_id}_clip_{i:03d}"})
    if problems:
        raise FileNotFoundError("; ".join(problems))
    if expected_counts is not None:
        if len(clips) != sum(expected_counts.values()):
            raise ValueError(f"Real clip total is {len(clips)}, expected {sum(expected_counts.values())}.")
        if counts != expected_counts:
            raise ValueError(f"Per-speaker clip counts {counts} differ from expected {expected_counts}.")
    return clips


def get_ffmpeg():
    """Path of an ffmpeg binary: the system one if present, otherwise the one bundled with imageio-ffmpeg (installed on demand); None if neither works."""
    exe = shutil.which("ffmpeg")
    if exe:
        return exe
    try:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "imageio-ffmpeg"], check=True)
        import imageio_ffmpeg
        return imageio_ffmpeg.get_ffmpeg_exe()
    except Exception:
        return None

In [ ]:
# Preflight: stop with every problem listed before any expensive step runs.
import torch, shutil, urllib.request, opensmile

print("MODE: SMOKE (not results)" if SMOKE else "MODE: FULL")
problems = []
if not torch.cuda.is_available():
    problems.append("No GPU detected: enable a GPU accelerator in the Kaggle notebook settings.")
if N_PAIRS != EXPECTED_N_PAIRS:
    problems.append(f"N_PAIRS must be {EXPECTED_N_PAIRS} in {'SMOKE' if SMOKE else 'FULL'} mode, got {N_PAIRS}.")
if CONTENT_LABEL_SOURCE not in ("chalearn", "proxy"):
    problems.append(f"CONTENT_LABEL_SOURCE must be 'chalearn' or 'proxy', got {CONTENT_LABEL_SOURCE!r}.")
if INDEPENDENCE_PENALTY not in ("xcorr", "dcor"):
    problems.append(f"INDEPENDENCE_PENALTY must be 'xcorr' or 'dcor', got {INDEPENDENCE_PENALTY!r}.")
FFMPEG_EXE = get_ffmpeg()
if FFMPEG_EXE is None:
    problems.append("ffmpeg not found on PATH and the imageio-ffmpeg fallback could not be installed.")
if shutil.disk_usage(WORK).free < 4e9:
    problems.append("Less than 4 GB free in /kaggle/working.")

CHALEARN_BASE = REAL_BASE = chalearn = real_clips = None
try:
    CHALEARN_BASE = find_dataset(DATASET_CHALEARN)
    chalearn = load_chalearn_training(CHALEARN_BASE, None if SMOKE else EXPECTED_CHALEARN_CLIPS)
    if SMOKE:
        chalearn = subset_chalearn(chalearn, SMOKE_CHALEARN_CLIPS, SMOKE_CHALEARN_MIN_VIDEOS, SEED)
        if not (250 <= chalearn["n_common"] <= 350 and len(set(chalearn["video_ids"])) >= SMOKE_CHALEARN_MIN_VIDEOS):
            problems.append(f"Smoke ChaLearn subset has {chalearn['n_common']} clips from {len(set(chalearn['video_ids']))} videos; expected about {SMOKE_CHALEARN_CLIPS} from at least {SMOKE_CHALEARN_MIN_VIDEOS}.")
except Exception as e:
    problems.append(f"ChaLearn dataset: {e}")
try:
    REAL_BASE = find_dataset(DATASET_REAL)
    wav_root = os.path.join(REAL_BASE, "wav")
    if SMOKE:
        real_clips = list_real_clips(wav_root, SPEAKER_FOLDER_MAP, None, max_per_speaker=SMOKE_REAL_PER_SPEAKER)
        want = sum(min(SMOKE_REAL_PER_SPEAKER, n) for n in EXPECTED_REAL_CLIPS.values())
        if len(real_clips) != want:
            problems.append(f"Smoke real-clip subset has {len(real_clips)} clips, expected {want}.")
        _full = {}
        for _c in list_real_clips(wav_root, SPEAKER_FOLDER_MAP, None):
            _full[_c["folder"]] = _full.get(_c["folder"], 0) + 1
        if _full != EXPECTED_REAL_CLIPS:
            print(f"NOTE (smoke): the real-clip folders hold {_full}, which differs from the expected full-run counts {EXPECTED_REAL_CLIPS}; a FULL run would fail its preflight.")
    else:
        real_clips = list_real_clips(wav_root, SPEAKER_FOLDER_MAP, EXPECTED_REAL_CLIPS)
except Exception as e:
    problems.append(f"Real clips dataset: {e}")

_smile_check = opensmile.Smile(feature_set=opensmile.FeatureSet.GeMAPSv01b, feature_level=opensmile.FeatureLevel.Functionals)
_names = list(_smile_check.feature_names)
if len(_names) != 62:
    problems.append(f"openSMILE GeMAPSv01b Functionals gave {len(_names)} features, expected 62.")
_missing = [f for f in AROUSAL_FEATURES if f not in _names]
if _missing:
    problems.append(f"Arousal-proxy features not found among GeMAPS names: {_missing}")

_libri_local = os.path.join(CHALEARN_BASE, "LibriSpeech", "test-clean") if CHALEARN_BASE else None
_libri_ok = bool(_libri_local and os.path.isdir(_libri_local) and any(f.endswith(".flac") for _, _, fs in os.walk(_libri_local) for f in fs))
if not _libri_ok:
    try:
        urllib.request.urlopen(urllib.request.Request("https://www.openslr.org/resources/12/test-clean.tar.gz", method="HEAD"), timeout=20)
    except Exception as e:
        problems.append(f"LibriSpeech test-clean is not in the ChaLearn dataset and cannot be downloaded: {e}")

if problems:
    raise RuntimeError("Preflight failed:\n- " + "\n- ".join(problems))
print("Preflight OK")
print(f"  ChaLearn: {CHALEARN_BASE} ({chalearn['n_common']} clips, {len(set(chalearn['video_ids']))} videos)")
print(f"  Real clips: {REAL_BASE} ({len(real_clips)} clips)")
print(f"  LibriSpeech: {'found in ChaLearn dataset' if _libri_ok else 'will be downloaded'}; ffmpeg: {FFMPEG_EXE}")
print(f"  Content label source: {CONTENT_LABEL_SOURCE}; independence penalty: {INDEPENDENCE_PENALTY}; LOSO retrain: {LOSO_RETRAIN}; outputs: {OUT_DIR}")

### A (continued). Shared model code

In [ ]:
# KEEP IN SYNC with src/api/autiz_model.py (copy of the block between its BEGIN/END SHARED MODEL CODE markers).
import torch
import torch.nn as nn

CONTENT_DIM = 1024
GEMAPS_DIM = 62
HIDDEN_DIM = 256


class ContentBranch(nn.Module):
    """RoBERTa CLS embedding -> C. Never sees audio."""

    def __init__(self, input_dim=CONTENT_DIM, hidden_dim=HIDDEN_DIM):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim),
        )

    def forward(self, x):
        return self.layers(x)


class ProsodyBranch(nn.Module):
    """z-scored GeMAPS features -> D. Never sees the transcript."""

    def __init__(self, input_dim=GEMAPS_DIM, hidden_dim=HIDDEN_DIM):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim),
        )

    def forward(self, x):
        return self.layers(x)


class ReconHead(nn.Module):
    """D -> reconstruction of the z-scored GeMAPS input. Training only."""

    def __init__(self, hidden_dim=HIDDEN_DIM, output_dim=GEMAPS_DIM):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, output_dim),
        )

    def forward(self, d):
        return self.layers(d)


def build_models(hidden_dim=HIDDEN_DIM):
    """Fresh (randomly initialised) modules for every part of the v2 model, keyed like the checkpoint."""
    return {
        "content_branch": ContentBranch(CONTENT_DIM, hidden_dim),
        "content_head": nn.Linear(hidden_dim, 1),
        "prosody_branch": ProsodyBranch(GEMAPS_DIM, hidden_dim),
        "recon_head": ReconHead(hidden_dim, GEMAPS_DIM),
    }


In [ ]:
# Statistics utilities (numpy and scikit-learn only): bias-corrected distance correlation, grouped ridge R2, permutation baselines.
import numpy as np
from scipy import stats as sps
from scipy.spatial.distance import cdist
from sklearn.linear_model import RidgeCV
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score

RIDGE_ALPHAS = np.logspace(-2, 4, 7)


def safe_pearson(a, b):
    """Pearson r, or NaN when either input is constant."""
    a, b = np.asarray(a, dtype=np.float64), np.asarray(b, dtype=np.float64)
    if a.std() < 1e-12 or b.std() < 1e-12:
        return float("nan")
    return float(np.corrcoef(a, b)[0, 1])


def r2_variance_weighted(y_true, y_pred):
    """R2 pooled over output columns (each column weighted by its variance)."""
    return float(r2_score(y_true, y_pred, multioutput="variance_weighted"))


def _u_centered(dist):
    n = dist.shape[0]
    u = dist - dist.sum(axis=1, keepdims=True) / (n - 2) - dist.sum(axis=0, keepdims=True) / (n - 2) + dist.sum() / ((n - 1) * (n - 2))
    np.fill_diagonal(u, 0.0)
    return u


def _dcor2_from_centered(a, b, n):
    denom = n * (n - 3)
    xy, xx, yy = (a * b).sum() / denom, (a * a).sum() / denom, (b * b).sum() / denom
    if xx <= 0 or yy <= 0:
        return float("nan")
    return float(xy / np.sqrt(xx * yy))


def dcor2_bias_corrected(X, Y):
    """Bias-corrected squared distance correlation (U-centering). About 0 for independent data; can be slightly negative."""
    X, Y = np.asarray(X, dtype=np.float64), np.asarray(Y, dtype=np.float64)
    n = len(X)
    assert n > 3 and len(Y) == n
    return _dcor2_from_centered(_u_centered(cdist(X, X)), _u_centered(cdist(Y, Y)), n)


def _null_summary(real, null):
    null = np.asarray(null, dtype=np.float64)
    return {
        "null_mean": float(null.mean()),
        "null_p95": float(np.percentile(null, 95)),
        "percentile_of_real_in_null": float(100.0 * (null < real).mean()),
        "p_value": float((1 + (null >= real).sum()) / (len(null) + 1)),
        "n_permutations": int(len(null)),
        "_null_values": null,
    }


def distance_correlation_report(X, Y, n_perm=200, seed=0):
    """Distance correlation with a permutation baseline (shuffle which Y row is paired with which X row)."""
    X, Y = np.asarray(X, dtype=np.float64), np.asarray(Y, dtype=np.float64)
    n = len(X)
    assert n > 3 and len(Y) == n
    a, b = _u_centered(cdist(X, X)), _u_centered(cdist(Y, Y))
    real = _dcor2_from_centered(a, b, n)
    rng = np.random.default_rng(seed)
    null = np.empty(n_perm)
    for i in range(n_perm):
        p = rng.permutation(n)
        null[i] = _dcor2_from_centered(a, b[np.ix_(p, p)], n)
    out = {"n": int(n), "dcor2_bias_corrected": real, "dcor_bias_corrected": float(np.sqrt(max(real, 0.0)))}
    out.update(_null_summary(real, null))
    return out


def _fit_predict_ridge(Xtr, Ytr, Xte):
    sc = StandardScaler().fit(Xtr)
    model = RidgeCV(alphas=RIDGE_ALPHAS).fit(sc.transform(Xtr), Ytr)
    return model.predict(sc.transform(Xte))


def grouped_ridge_r2(X, Y, groups, n_splits=5):
    """Out-of-fold R2 of a ridge regression X -> Y with cross-validation grouped by `groups` (e.g. speaker)."""
    X, Y, groups = np.asarray(X, dtype=np.float64), np.asarray(Y, dtype=np.float64), np.asarray(groups)
    k = min(n_splits, len(np.unique(groups)))
    assert k >= 2, "need at least 2 groups for grouped cross-validation"
    oof = np.zeros_like(Y)
    for tr, te in GroupKFold(n_splits=k).split(X, Y, groups):
        oof[te] = _fit_predict_ridge(X[tr], Y[tr], X[te])
    return r2_variance_weighted(Y, oof)


def ridge_report(X, Y, groups, n_perm=200, seed=0):
    """Grouped cross-validated ridge R2 (X -> Y) with a permutation baseline (shuffle the X-Y pairing)."""
    real = grouped_ridge_r2(X, Y, groups)
    rng = np.random.default_rng(seed)
    Y = np.asarray(Y, dtype=np.float64)
    null = np.array([grouped_ridge_r2(X, Y[rng.permutation(len(Y))], groups) for _ in range(n_perm)])
    out = {"n": int(len(Y)), "r2": real}
    out.update(_null_summary(real, null))
    return out


def ridge_transfer_report(Xtr, Ytr, Xte, Yte, n_perm=200, seed=0):
    """Ridge fitted on a training set and scored on held-out rows, with a permutation baseline on the held-out pairing."""
    Yte = np.asarray(Yte, dtype=np.float64)
    pred = _fit_predict_ridge(np.asarray(Xtr, dtype=np.float64), np.asarray(Ytr, dtype=np.float64), np.asarray(Xte, dtype=np.float64))
    real = r2_variance_weighted(Yte, pred)
    rng = np.random.default_rng(seed)
    null = np.array([r2_variance_weighted(Yte[rng.permutation(len(Yte))], pred) for _ in range(n_perm)])
    out = {"n_train": int(len(Xtr)), "n_test": int(len(Xte)), "r2": real}
    out.update(_null_summary(real, null))
    return out


def cohens_d_paired(a, b):
    """Cohen's d for paired samples: mean difference divided by the standard deviation of the differences."""
    diff = np.asarray(a, dtype=np.float64) - np.asarray(b, dtype=np.float64)
    sd = diff.std(ddof=1)
    return float(diff.mean() / sd) if sd > 1e-12 else float("nan")


def to_jsonable(obj):
    """Convert numpy values to plain Python, drop keys starting with '_', and turn NaN/inf into None."""
    if isinstance(obj, dict):
        return {str(k): to_jsonable(v) for k, v in obj.items() if not str(k).startswith("_")}
    if isinstance(obj, (list, tuple)):
        return [to_jsonable(v) for v in obj]
    if isinstance(obj, np.ndarray):
        return to_jsonable(obj.tolist())
    if isinstance(obj, (np.floating, float)):
        return float(obj) if np.isfinite(obj) else None
    if isinstance(obj, np.integer):
        return int(obj)
    if isinstance(obj, np.bool_):
        return bool(obj)
    return obj
def write_json(path, obj):
    """Write obj as JSON (NaN -> null); in smoke mode the first key is run_stamp."""
    obj = dict(obj)
    if RUN_STAMP:
        obj = {"run_stamp": RUN_STAMP, **obj}
    with open(path, "w", encoding="utf-8") as f:
        json.dump(to_jsonable(obj), f, indent=2)


def write_csv(df, path):
    """Write a DataFrame as CSV; in smoke mode the first line is a '# SMOKE RUN, NOT RESULTS' comment."""
    with open(path, "w", encoding="utf-8", newline="") as f:
        if RUN_STAMP:
            f.write(f"# {RUN_STAMP}\n")
        df.to_csv(f, index=False, lineterminator="\n")

In [ ]:
# Torch utilities: seeding, independence penalties, the two training loops, batched encoding.
import torch
import torch.nn as nn
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def xcorr_penalty(C, D, eps=1e-4):
    """Mean squared cross-correlation between the dimensions of C and D over a batch (0 when linearly uncorrelated)."""
    n = C.shape[0]
    Cn = (C - C.mean(0, keepdim=True)) / (C.std(0, keepdim=True) + eps)
    Dn = (D - D.mean(0, keepdim=True)) / (D.std(0, keepdim=True) + eps)
    return ((Cn.T @ Dn / (n - 1)) ** 2).mean()


def dcor_penalty(C, D, eps=1e-8):
    """Differentiable (biased) squared distance correlation between C and D over a batch."""
    def centered(X):
        d = torch.sqrt((X.unsqueeze(1) - X.unsqueeze(0)).pow(2).sum(-1) + eps)
        return d - d.mean(0, keepdim=True) - d.mean(1, keepdim=True) + d.mean()
    a, b = centered(C), centered(D)
    return (a * b).mean() / (torch.sqrt((a * a).mean() * (b * b).mean()) + eps)


def independence_penalty(C, D, kind):
    """Batch-level independence penalty between frozen C and trainable D."""
    return xcorr_penalty(C, D) if kind == "xcorr" else dcor_penalty(C, D)


def encode(module, X, batch_size=1024):
    """Run a module over a numpy array in batches (eval mode, no gradients) and return a numpy array."""
    module.eval()
    out = []
    with torch.no_grad():
        for i in range(0, len(X), batch_size):
            xb = torch.as_tensor(np.asarray(X[i:i + batch_size]), dtype=torch.float32, device=device)
            out.append(module(xb).cpu().numpy())
    return np.concatenate(out)


def _clone_state(*modules):
    return [{k: v.detach().clone() for k, v in m.state_dict().items()} for m in modules]


def train_content(Xtr, ytr, Xes, yes, hp, log):
    """Stage 1: train ContentBranch + content_head with MSE; early stopping on a held-out set; returns frozen modules."""
    set_seed(SEED)
    models = build_models()
    branch, head = models["content_branch"].to(device), models["content_head"].to(device)
    params = list(branch.parameters()) + list(head.parameters())
    opt = torch.optim.AdamW(params, lr=hp["lr"], weight_decay=hp["weight_decay"])
    Xtr_t, ytr_t = torch.as_tensor(Xtr, dtype=torch.float32, device=device), torch.as_tensor(ytr, dtype=torch.float32, device=device)
    Xes_t, yes_t = torch.as_tensor(Xes, dtype=torch.float32, device=device), torch.as_tensor(yes, dtype=torch.float32, device=device)
    gen = torch.Generator(device=device)
    gen.manual_seed(SEED)
    best_mse, best_epoch, best_state, wait = float("inf"), 0, None, 0
    for epoch in range(1, hp["max_epochs"] + 1):
        branch.train(); head.train()
        perm = torch.randperm(len(Xtr_t), device=device, generator=gen)
        total = 0.0
        for i in range(0, len(perm), hp["batch_size"]):
            idx = perm[i:i + hp["batch_size"]]
            loss = F.mse_loss(head(branch(Xtr_t[idx])).squeeze(-1), ytr_t[idx])
            opt.zero_grad(); loss.backward(); opt.step()
            total += loss.item() * len(idx)
        branch.eval(); head.eval()
        with torch.no_grad():
            pred = head(branch(Xes_t)).squeeze(-1)
            es_mse = F.mse_loss(pred, yes_t).item()
        log.append({"stage": "content", "run": "content", "epoch": epoch, "train_mse": total / len(Xtr_t), "es_mse": es_mse,
                    "es_pearson_r": safe_pearson(yes, pred.cpu().numpy())})
        if es_mse < best_mse - 1e-6:
            best_mse, best_epoch, best_state, wait = es_mse, epoch, _clone_state(branch, head), 0
        else:
            wait += 1
            if wait >= hp["patience"]:
                break
    branch.load_state_dict(best_state[0]); head.load_state_dict(best_state[1])
    for p in params:
        p.requires_grad_(False)
    branch.eval(); head.eval()
    return branch, head, best_epoch


def train_prosody(C_fit, Z_fit, lam, max_epochs, hp, init_seed, shuffle_seed, run_id, log, val=None):
    """Stage 2: train ProsodyBranch + recon_head with reconstruction MSE + lam * independence penalty against frozen C.

    With `val=(C_val, Z_val)` the best epoch (lowest validation objective) is kept; without it, training runs
    for exactly `max_epochs`. Returns (prosody_branch, recon_head, epochs_used)."""
    torch.manual_seed(init_seed)
    models = build_models()
    pb, rh = models["prosody_branch"].to(device), models["recon_head"].to(device)
    params = list(pb.parameters()) + list(rh.parameters())
    opt = torch.optim.AdamW(params, lr=hp["lr"], weight_decay=hp["weight_decay"])
    C_t, Z_t = torch.as_tensor(C_fit, dtype=torch.float32, device=device), torch.as_tensor(Z_fit, dtype=torch.float32, device=device)
    if val is not None:
        C_v, Z_v = torch.as_tensor(val[0], dtype=torch.float32, device=device), torch.as_tensor(val[1], dtype=torch.float32, device=device)
    assert len(C_t) >= MIN_BATCH_ROWS, f"need at least {MIN_BATCH_ROWS} training rows"
    gen = torch.Generator(device=device)
    gen.manual_seed(shuffle_seed)
    best_obj, best_epoch, best_state, wait = float("inf"), max_epochs, None, 0
    for epoch in range(1, max_epochs + 1):
        pb.train(); rh.train()
        perm = torch.randperm(len(C_t), device=device, generator=gen)
        rec_sum = ind_sum = 0.0
        n_seen = 0
        for i in range(0, len(perm), hp["batch_size"]):
            idx = perm[i:i + hp["batch_size"]]
            if len(idx) < MIN_BATCH_ROWS:
                continue
            D = pb(Z_t[idx])
            l_rec = F.mse_loss(rh(D), Z_t[idx])
            l_ind = independence_penalty(C_t[idx], D, INDEPENDENCE_PENALTY)
            loss = l_rec + lam * l_ind
            opt.zero_grad(); loss.backward(); opt.step()
            rec_sum += l_rec.item() * len(idx); ind_sum += l_ind.item() * len(idx); n_seen += len(idx)
        row = {"stage": "prosody", "run": run_id, "epoch": epoch, "lambda": lam,
               "train_recon_mse": rec_sum / n_seen, "train_indep_penalty": ind_sum / n_seen}
        if val is not None:
            pb.eval(); rh.eval()
            with torch.no_grad():
                Dv = pb(Z_v)
                v_rec = F.mse_loss(rh(Dv), Z_v).item()
                v_ind = independence_penalty(C_v, Dv, INDEPENDENCE_PENALTY).item()
            obj = v_rec + lam * v_ind
            row.update({"val_recon_mse": v_rec, "val_indep_penalty": v_ind, "val_objective": obj})
            if obj < best_obj - 1e-6:
                best_obj, best_epoch, best_state, wait = obj, epoch, _clone_state(pb, rh), 0
            else:
                wait += 1
        log.append(row)
        if val is not None and wait >= hp["patience"]:
            break
    if val is not None and best_state is not None:
        pb.load_state_dict(best_state[0]); rh.load_state_dict(best_state[1])
    for p in params:
        p.requires_grad_(False)
    pb.eval(); rh.eval()
    return pb, rh, (best_epoch if val is not None else max_epochs)

In [ ]:
# Self-test on random data: runs every function with tiny inputs so a bug shows up here, before any real work starts.
_rng = np.random.default_rng(0)
_X = _rng.normal(size=(200, 8))
_Y_dep = _X @ _rng.normal(size=(8, 8)) + 0.1 * _rng.normal(size=(200, 8))
_Y_ind = _rng.normal(size=(200, 8))
_g = np.repeat(np.arange(10), 20)

_dep = distance_correlation_report(_X, _Y_dep, n_perm=50, seed=0)
_ind = distance_correlation_report(_X, _Y_ind, n_perm=50, seed=0)
assert _dep["p_value"] < 0.05 and _dep["dcor2_bias_corrected"] > 0.2, _dep
assert abs(_ind["dcor2_bias_corrected"]) < 0.1, _ind
_r_dep = ridge_report(_X, _Y_dep, _g, n_perm=10, seed=0)
_r_ind = ridge_report(_X, _Y_ind, _g, n_perm=10, seed=0)
assert _r_dep["r2"] > 0.5 and _r_ind["r2"] < 0.2, (_r_dep["r2"], _r_ind["r2"])
_t = ridge_transfer_report(_X[:150], _Y_dep[:150], _X[150:], _Y_dep[150:], n_perm=10, seed=0)
assert _t["r2"] > 0.5, _t["r2"]
assert math.isnan(safe_pearson([1, 1, 1], [1, 2, 3]))

_C = _rng.normal(size=(300, 1024)).astype(np.float32)
_y = _rng.uniform(size=300).astype(np.float32)
_tmp_log = []
_cb, _ch, _ep = train_content(_C[:200], _y[:200], _C[200:], _y[200:], dict(CONTENT_HP, max_epochs=3), _tmp_log)
_Cenc = encode(_cb, _C)
assert _Cenc.shape == (300, 256) and np.isfinite(_Cenc).all()
_Z = _rng.normal(size=(300, 62)).astype(np.float32)
for _kind in ("xcorr", "dcor"):
    INDEPENDENCE_PENALTY, _saved = _kind, INDEPENDENCE_PENALTY
    _pb, _rh, _used = train_prosody(_Cenc[:200], _Z[:200], 1.0, 3, dict(PROSODY_HP), 1, 2, f"selftest_{_kind}", _tmp_log, val=(_Cenc[200:], _Z[200:]))
    _D = encode(_pb, _Z)
    assert _D.shape == (300, 256) and np.isfinite(_D).all() and np.isfinite(encode(_rh, _D)).all()
    INDEPENDENCE_PENALTY = _saved
_pb2, _rh2, _used2 = train_prosody(_Cenc[:200], _Z[:200], 0.0, 2, dict(PROSODY_HP), 1, 2, "selftest_noval", _tmp_log)
assert _used2 == 2

_models = build_models()
_path = os.path.join(tempfile.mkdtemp(), "selftest.pt")
torch.save({"content_branch": _cb.state_dict(), "content_head": _ch.state_dict(), "prosody_branch": _pb.state_dict(),
            "recon_head": _rh.state_dict(), "config": {"dims": {"hidden": 256}, "seed": 0}}, _path)
_loaded = torch.load(_path, map_location="cpu", weights_only=True)
_models["content_branch"].load_state_dict(_loaded["content_branch"])
assert set(_loaded) == {"content_branch", "content_head", "prosody_branch", "recon_head", "config"}
del _cb, _ch, _pb, _rh, _pb2, _rh2, _models, _loaded
print("Self-test passed: statistics, both penalties, both training loops, checkpoint round trip")
section_end("A. Setup, preflight, shared code and self-test")

## B. Synthetic pairs (500), split by LibriSpeech speaker

In [ ]:
# LibriSpeech: use the copy inside the ChaLearn dataset if present, otherwise download test-clean; read the ground-truth transcripts.
section_start("B. Synthetic pairs")
import tarfile, urllib.request
import soundfile as sf


def flac_files(root):
    return sorted(os.path.join(d, f) for d, _, fs in os.walk(root) for f in fs if f.endswith(".flac"))


LIBRI_ROOT = os.path.join(CHALEARN_BASE, "LibriSpeech", "test-clean")
if not (os.path.isdir(LIBRI_ROOT) and flac_files(LIBRI_ROOT)):
    tgz = f"{WORK}/test-clean.tar.gz"
    urllib.request.urlretrieve("https://www.openslr.org/resources/12/test-clean.tar.gz", tgz)
    with tarfile.open(tgz, "r:gz") as tar:
        try:
            tar.extractall(WORK, filter="data")
        except TypeError:
            tar.extractall(WORK)
    LIBRI_ROOT = f"{WORK}/LibriSpeech/test-clean"

libri_flac = flac_files(LIBRI_ROOT)
assert libri_flac, f"no .flac files under {LIBRI_ROOT}"
libri_text = {}
for d, _, fs in os.walk(LIBRI_ROOT):
    for f in fs:
        if f.endswith(".trans.txt"):
            with open(os.path.join(d, f), encoding="utf-8") as fh:
                for line in fh:
                    uid, _, txt = line.strip().partition(" ")
                    libri_text[uid] = txt


def utt_id(path):
    return os.path.splitext(os.path.basename(path))[0]


def libri_to_text(raw):
    """LibriSpeech references are upper case; use sentence case so they look like ordinary text to RoBERTa."""
    t = raw.lower()
    return t[:1].upper() + t[1:]


assert not [p for p in libri_flac if utt_id(p) not in libri_text], "some LibriSpeech clips have no transcript"
eligible = [p for p in libri_flac if sf.info(p).duration < MAX_PAIR_SECONDS]
print(f"LibriSpeech: {LIBRI_ROOT}: {len(libri_flac)} clips, {len(eligible)} shorter than {MAX_PAIR_SECONDS:.0f} s, "
      f"{len({utt_id(p).split('-')[0] for p in eligible})} speakers")

In [ ]:
# Synthetic pairs: render an original and a pitch-flattened, 15 percent slower copy of 500 eligible clips; stale pairs inside the dataset are ignored.
import parselmouth
from parselmouth.praat import call


def render_pair(src_path, out_original, out_flat):
    """Write a 16 kHz original and a flattened-and-stretched copy of src_path; return the duration in seconds."""
    sound = parselmouth.Sound(src_path)
    pitch_obj = call(sound, "To Pitch", 0.0, 75, 600)
    mean_pitch = call(pitch_obj, "Get mean", 0, 0, "Hertz")
    if mean_pitch is None or np.isnan(mean_pitch) or mean_pitch <= 0:
        mean_pitch = 120.0
    manipulation = call(sound, "To Manipulation", 0.01, 75, 600)
    pitch_tier = call(manipulation, "Extract pitch tier")
    call(pitch_tier, "Remove points between", 0, sound.duration)
    call(pitch_tier, "Add point", sound.duration / 2, mean_pitch * 0.95)
    call([pitch_tier, manipulation], "Replace pitch tier")
    duration_tier = call("Create DurationTier", "dur", 0, sound.duration)
    call(duration_tier, "Add point", sound.duration / 2, 1.15)
    call([duration_tier, manipulation], "Replace duration tier")
    flat_sound = call(manipulation, "Get resynthesis (overlap-add)")
    call(call(sound, "Resample", 16000, 50), "Save as WAV file", out_original)
    call(call(flat_sound, "Resample", 16000, 50), "Save as WAV file", out_flat)
    return sound.duration


shutil.rmtree(PAIRS_DIR, ignore_errors=True)
os.makedirs(PAIRS_DIR)
candidates = list(eligible)
random.Random(SEED).shuffle(candidates)
if SMOKE:
    by_spk = {}
    for p in candidates:
        by_spk.setdefault(utt_id(p).split("-")[0], []).append(p)
    spk_pick = random.Random(SEED).sample(sorted(by_spk), 10)
    candidates = [by_spk[sp][r] for r in range(3) for sp in spk_pick if len(by_spk[sp]) > r]
pair_records, failed_pairs = [], []
for src in candidates:
    if len(pair_records) == N_PAIRS:
        break
    k = len(pair_records)
    out_o, out_f = f"{PAIRS_DIR}/pair_{k:03d}_original.wav", f"{PAIRS_DIR}/pair_{k:03d}_flat.wav"
    try:
        dur = render_pair(src, out_o, out_f)
    except Exception as e:
        failed_pairs.append((src, str(e)))
        continue
    pair_records.append({"pair_id": f"pair_{k:03d}", "src": src, "speaker": utt_id(src).split("-")[0], "text": libri_to_text(libri_text[utt_id(src)]),
                         "path_orig": out_o, "path_flat": out_f, "duration": dur})
assert len(pair_records) == N_PAIRS, f"only {len(pair_records)} pairs rendered; {len(failed_pairs)} clips failed"
print(f"Rendered {len(pair_records)} pairs ({len(failed_pairs)} candidate clips failed and were skipped)")

In [ ]:
# Split the pairs by LibriSpeech speaker: 80 percent train side (20 percent of it validation) and 20 percent held-out test speakers.
pair_speakers = np.array([r["speaker"] for r in pair_records])
all_speakers = sorted(set(pair_speakers))
shuffled = [str(s) for s in np.random.default_rng(SEED).permutation(all_speakers)]
n_test = max(1, round(HELD_OUT_FRACTION * len(shuffled)))
test_speakers, rest = set(shuffled[:n_test]), shuffled[n_test:]
n_val = max(1, round(VAL_FRACTION_OF_TRAIN * len(rest)))
val_speakers, train_speakers = set(rest[:n_val]), set(rest[n_val:])
assert not (test_speakers & val_speakers) and not (test_speakers & train_speakers) and not (val_speakers & train_speakers)

split = np.array(["test" if s in test_speakers else "val" if s in val_speakers else "train" for s in pair_speakers])
pairs = {
    "pair_id": [r["pair_id"] for r in pair_records],
    "speaker": pair_speakers,
    "split": split,
    "text": [r["text"] for r in pair_records],
    "path_orig": [r["path_orig"] for r in pair_records],
    "path_flat": [r["path_flat"] for r in pair_records],
}
for name, spk in (("train", train_speakers), ("val", val_speakers), ("test", test_speakers)):
    print(f"  {name:5s}: {len(spk):2d} speakers, {(split == name).sum():3d} pairs")
assert len(test_speakers) >= MIN_TEST_SPEAKERS and (split == "test").sum() >= MIN_TEST_PAIRS, "held-out split too small for grouped cross-validation"
section_end("B. Synthetic pairs")

## C. GeMAPS features and the z-score scaler

In [ ]:
# GeMAPS: convert real clips to 16 kHz mono, extract the 62 features once for every audio file, cache them and validate.
section_start("C. GeMAPS features and scaler")
import opensmile

for c in real_clips:
    if not os.path.isfile(c["src"]):
        raise FileNotFoundError(f"Real clip {c['clip_id']} was listed but is not on disk: {c['src']}")
    dst_dir = os.path.join(REAL_WAV_DIR, c["speaker_id"])
    os.makedirs(dst_dir, exist_ok=True)
    c["wav"] = os.path.join(dst_dir, c["clip_id"] + ".wav")
    res = subprocess.run([FFMPEG_EXE, "-y", "-i", c["src"], "-ar", "16000", "-ac", "1", c["wav"], "-loglevel", "error"], capture_output=True, text=True)
    if res.returncode != 0 or not os.path.exists(c["wav"]) or os.path.getsize(c["wav"]) == 0:
        raise RuntimeError(f"ffmpeg failed on {c['clip_id']} ({c['folder']}): {res.stderr.strip()[:300]}")

smile = opensmile.Smile(feature_set=opensmile.FeatureSet.GeMAPSv01b, feature_level=opensmile.FeatureLevel.Functionals)
FEATURE_NAMES = list(smile.feature_names)
assert len(FEATURE_NAMES) == 62

all_files = list(pairs["path_orig"]) + list(pairs["path_flat"]) + [c["wav"] for c in real_clips]
signature = hashlib.sha1("\n".join(sorted([r["src"] for r in pair_records] + [c["src"] for c in real_clips])).encode()).hexdigest()
cache = {}
if os.path.exists(OUT["gemaps_cache"]):
    z = np.load(OUT["gemaps_cache"], allow_pickle=False)
    if str(z["signature"]) == signature:
        cache = dict(zip(z["paths"].tolist(), z["features"]))
for p in all_files:
    if p not in cache:
        if not os.path.isfile(p):
            raise FileNotFoundError(f"Audio file missing before GeMAPS extraction: {p}")
        cache[p] = smile.process_file(p).values[0].astype(np.float32)
np.savez(OUT["gemaps_cache"], signature=signature, paths=np.array(list(cache)), features=np.stack([cache[p] for p in cache]))

bad = [p for p in all_files if cache[p].shape != (62,) or not np.isfinite(cache[p]).all()]
assert not bad, f"{len(bad)} files have a GeMAPS vector that is not 62 finite values, e.g. {bad[:3]}"
pairs["gem_orig"] = np.stack([cache[p] for p in pairs["path_orig"]])
pairs["gem_flat"] = np.stack([cache[p] for p in pairs["path_flat"]])
real_gem = np.stack([cache[c["wav"]] for c in real_clips])
print(f"GeMAPS: {len(all_files)} files, 62 columns each, all finite")

In [ ]:
# Scaler: fit mean and std on ORIGINAL clips of the training split only, save gemaps_scaler.json, z-score every feature set.
train_side = pairs["split"] != "test"
fit_X = pairs["gem_orig"][train_side].astype(np.float64)
scaler_mean = fit_X.mean(axis=0)
scaler_std_raw = fit_X.std(axis=0)
scaler_std = np.where(scaler_std_raw < STD_FLOOR_THRESHOLD, STD_FLOOR_VALUE, scaler_std_raw)
floored_features = [FEATURE_NAMES[i] for i in np.where(scaler_std_raw < STD_FLOOR_THRESHOLD)[0]]


def zscore_raw(x):
    """z-score with the training-split scaler, no clipping."""
    return (np.asarray(x, dtype=np.float64) - scaler_mean) / scaler_std


def zscore(x):
    """z-score with the training-split scaler and clip to +-Z_CLIP (what the model sees)."""
    return np.clip(zscore_raw(x), -Z_CLIP, Z_CLIP).astype(np.float32)


write_json(OUT["scaler"], {
    "feature_names": FEATURE_NAMES, "mean": scaler_mean.tolist(), "std": scaler_std.tolist(), "z_clip": Z_CLIP,
    "std_floor_threshold": STD_FLOOR_THRESHOLD, "std_floor_value": STD_FLOOR_VALUE, "features_with_std_floor_applied": floored_features,
    "fit_on": "ORIGINAL (unflattened) LibriSpeech test-clean pair clips of the training-split speakers only",
    "n_fit_clips": int(train_side.sum()), "date": RUN_DATE})

pairs["z_orig"], pairs["z_flat"] = zscore(pairs["gem_orig"]), zscore(pairs["gem_flat"])
real_z, real_zraw = zscore(real_gem), zscore_raw(real_gem)
for name, arr in (("train-side originals", zscore_raw(pairs["gem_orig"][train_side])), ("test originals", zscore_raw(pairs["gem_orig"][~train_side])),
                  ("flattened pairs", zscore_raw(pairs["gem_flat"])), ("real clips", real_zraw)):
    print(f"  share of values clipped at +-{Z_CLIP:.0f} SD, {name}: {100 * (np.abs(arr) > Z_CLIP).mean():.2f}%")
section_end("C. GeMAPS features and scaler")

## D. Stage 1 — content scorer (C), trained on ChaLearn transcripts

In [ ]:
# Stage 1 inputs: load frozen RoBERTa-large, define the cached CLS-embedding function, embed the ChaLearn transcripts.
section_start("D. Stage 1 content scorer")
from transformers import RobertaModel, RobertaTokenizer

roberta_tok = RobertaTokenizer.from_pretrained(ROBERTA_NAME)
roberta = RobertaModel.from_pretrained(ROBERTA_NAME).to(device).eval()
for _p in roberta.parameters():
    _p.requires_grad_(False)


def embed_texts(name, texts, batch_size=64):
    """CLS embeddings [N, 1024] from frozen RoBERTa-large, cached as .npy keyed by a hash of the texts."""
    key = hashlib.sha1("\n".join(texts).encode("utf-8")).hexdigest()[:10]
    path = f"{OUT_DIR}/cls_{name}_{key}.npy"
    if os.path.exists(path):
        arr = np.load(path)
        assert arr.shape == (len(texts), 1024), f"{path} has shape {arr.shape}"
        return arr
    order = np.argsort([len(t) for t in texts])
    out = np.zeros((len(texts), 1024), dtype=np.float32)
    with torch.inference_mode():
        for i in range(0, len(texts), batch_size):
            idx = order[i:i + batch_size]
            enc = roberta_tok([texts[j] for j in idx], return_tensors="pt", padding=True, truncation=True, max_length=512).to(device)
            out[idx] = roberta(**enc).last_hidden_state[:, 0, :].float().cpu().numpy()
    np.save(path, out)
    return out


cls_chalearn = embed_texts("chalearn", chalearn["texts"])
print("ChaLearn CLS embeddings:", cls_chalearn.shape)

In [ ]:
# Stage 1 training: split ChaLearn by YouTube video id, train ContentBranch + content_head on the interview score, freeze afterwards.
labels, video_ids = chalearn["labels"], chalearn["video_ids"]
train_idx, val_idx = group_split(video_ids, 0.2, SEED)
core_rel, es_rel = group_split(video_ids[train_idx], 0.1, SEED + 1)
core_idx, es_idx = train_idx[core_rel], train_idx[es_rel]

chalearn_sets = {"train core": core_idx, "early-stopping": es_idx, "validation (reported)": val_idx}
for name, idx in chalearn_sets.items():
    print(f"  {name:22s}: {len(set(video_ids[idx])):4d} unique videos, {len(idx):4d} clips")
for (na, a), (nb, b) in [(("train core", core_idx), ("validation", val_idx)), (("train core", core_idx), ("early-stopping", es_idx)),
                         (("early-stopping", es_idx), ("validation", val_idx))]:
    assert not (set(video_ids[a]) & set(video_ids[b])), f"video id overlap between {na} and {nb}"
assert len(set(video_ids[train_idx]) & set(video_ids[val_idx])) == 0

if CONTENT_LABEL_SOURCE == "chalearn":
    content_branch, content_head, content_best_epoch = train_content(
        cls_chalearn[core_idx], labels[core_idx], cls_chalearn[es_idx], labels[es_idx], CONTENT_HP, log_rows)
    print(f"Stage 1 done: best epoch {content_best_epoch}")
else:
    set_seed(SEED)
    _m = build_models()
    content_branch, content_head, content_best_epoch = _m["content_branch"].to(device).eval(), _m["content_head"].to(device).eval(), 0
    for _p in list(content_branch.parameters()) + list(content_head.parameters()):
        _p.requires_grad_(False)
    print(f"Stage 1 SKIPPED ({PROXY_STAMP}): ContentBranch is randomly initialised and frozen")

In [ ]:
# Stage 1 evaluation on held-out videos: Pearson r with a video-level bootstrap interval; save eval_results_content.json.
def content_scores(cls_arr):
    """(C [N,256], score [N]) from the frozen content path."""
    C = encode(content_branch, cls_arr)
    return C, encode(content_head, C).squeeze(-1)


stamp = PROXY_STAMP if CONTENT_LABEL_SOURCE == "proxy" else None
_, pred_val = content_scores(cls_chalearn[val_idx])
y_val, g_val = labels[val_idx], video_ids[val_idx]
by_video = {v: np.where(g_val == v)[0] for v in np.unique(g_val)}
_rng = np.random.default_rng(SEED)
boot = []
for _ in range(1000):
    pick = _rng.choice(list(by_video), size=len(by_video), replace=True)
    ii = np.concatenate([by_video[v] for v in pick])
    boot.append(safe_pearson(y_val[ii], pred_val[ii]))
train_mean = labels[core_idx].mean()
content_eval = {
    "content_label_source": CONTENT_LABEL_SOURCE,
    "content_scorer_stamp": stamp,
    "label_used": "ChaLearn First Impressions V2 'interview' score (human impression of a video clip)",
    "n_validation_clips": int(len(val_idx)),
    "n_validation_videos": int(len(by_video)),
    "best_epoch": int(content_best_epoch),
    "pearson_r_validation": safe_pearson(y_val, pred_val),
    "pearson_r_bootstrap_95ci_by_video": [float(np.nanpercentile(boot, 2.5)), float(np.nanpercentile(boot, 97.5))],
    "spearman_rho_validation": float(sps.spearmanr(y_val, pred_val)[0]),
    "mse_validation": float(np.mean((y_val - pred_val) ** 2)),
    "mse_predict_train_mean_baseline": float(np.mean((y_val - train_mean) ** 2)),
    "original_spec_target_pearson_r": 0.75,
    "plain_language": {
        "pearson_r_validation": "How well the text-only score tracks the human 'interview' impression on videos the model never saw. 1 is perfect, 0 is no relationship.",
        "caveat": "The labels are human impressions of video, not ratings of answer quality, so this r measures agreement with a proxy. The 0.75 target in the original spec was for 500 manually annotated transcripts, a different task.",
    },
}
write_json(OUT["content"], content_eval)
print(f"Held-out videos: Pearson r = {content_eval['pearson_r_validation']:.3f}, 95% CI {content_eval['pearson_r_bootstrap_95ci_by_video']}, "
      f"MSE {content_eval['mse_validation']:.4f} vs predict-the-mean {content_eval['mse_predict_train_mean_baseline']:.4f}")
section_end("D. Stage 1 content scorer")

## E. Stage 2 — delivery representation (D), reconstruction + independence from frozen C

In [ ]:
# Stage 2 inputs: C for every synthetic pair from its ground-truth transcript, and the train / validation / test arrays.
section_start("E. Stage 2 delivery representation")
cls_pairs = embed_texts("libri_pairs", pairs["text"])
C_pairs, score_pairs = content_scores(cls_pairs)
assert C_pairs.shape == (N_PAIRS, 256) and np.isfinite(C_pairs).all()

idx_train = np.where(pairs["split"] == "train")[0]
idx_val = np.where(pairs["split"] == "val")[0]
idx_test = np.where(pairs["split"] == "test")[0]


def stack_orig_flat(idx):
    """Original and flattened copies share the same C (same transcript) but have different prosody."""
    return np.concatenate([C_pairs[idx], C_pairs[idx]]), np.concatenate([pairs["z_orig"][idx], pairs["z_flat"][idx]])


C_tr, Z_tr = stack_orig_flat(idx_train)
C_va, Z_va = stack_orig_flat(idx_val)
print(f"Stage 2 rows: train {len(C_tr)}, validation {len(C_va)}, held-out test pairs {len(idx_test)}")

In [ ]:
# Stage 2 lambda sweep: train one ProsodyBranch per lambda (0.0 is the no-penalty ablation), select on validation speakers, never on test.
sweep, sweep_models = [], {}
for lam in LAMBDA_GRID:
    pb, rh, used = train_prosody(C_tr, Z_tr, lam, PROSODY_HP["max_epochs"], PROSODY_HP, init_seed=SEED + 11, shuffle_seed=SEED + 12,
                                 run_id=f"stage2_lambda_{lam}", log=log_rows, val=(C_va, Z_va))
    D_va = encode(pb, Z_va)
    val_recon_r2 = r2_variance_weighted(Z_va, encode(rh, D_va))
    D_va_orig = encode(pb, pairs["z_orig"][idx_val])
    val_dcor2 = dcor2_bias_corrected(C_pairs[idx_val], D_va_orig)
    select_score = (1.0 - val_recon_r2) + float(np.sqrt(max(val_dcor2, 0.0)))
    sweep.append({"lambda": lam, "best_epoch": int(used), "val_recon_r2": val_recon_r2, "val_dcor2_bias_corrected": val_dcor2,
                  "selection_score_lower_is_better": select_score})
    sweep_models[lam] = (pb, rh)
    print(f"  lambda={lam:>6}: epochs {used:3d}, val recon R2 {val_recon_r2:.3f}, val dCor2 {val_dcor2:+.4f}, score {select_score:.3f}")

selected_lambda = min((r for r in sweep if r["lambda"] > 0), key=lambda r: r["selection_score_lower_is_better"])["lambda"]
pb_final, rh_final = sweep_models[selected_lambda]
pb_none, rh_none = sweep_models[0.0]
best_epoch_final = next(r["best_epoch"] for r in sweep if r["lambda"] == selected_lambda)
D_orig_all, D_flat_all = encode(pb_final, pairs["z_orig"]), encode(pb_final, pairs["z_flat"])
print(f"Selected lambda = {selected_lambda} (epochs used {best_epoch_final})")
section_end("E. Stage 2 delivery representation")

## F. Save checkpoint and training log

In [ ]:
# Save autiz_v2.pt and training_log_v2.csv, then reload the checkpoint and confirm it reproduces the in-memory outputs.
section_start("F. Save checkpoint and log")
config = {
    "dims": {"content_in": CONTENT_DIM, "hidden": HIDDEN_DIM, "gemaps": GEMAPS_DIM},
    "seed": SEED,
    "content_label_source": CONTENT_LABEL_SOURCE,
    "content_scorer_stamp": stamp,
    "date": RUN_DATE,
    "selected_lambda": float(selected_lambda),
    "independence_penalty": INDEPENDENCE_PENALTY,
    "z_clip": float(Z_CLIP),
    "trained_on": "Stage 1: ChaLearn First Impressions V2 training transcripts; Stage 2: synthetic LibriSpeech pairs (train speakers). Real clips never used for this checkpoint.",
    "roberta": ROBERTA_NAME,
    "roberta_frozen": True,
}
if RUN_STAMP:
    config["run_stamp"] = RUN_STAMP


def cpu_state(module):
    return {k: v.detach().cpu().clone() for k, v in module.state_dict().items()}


torch.save({"content_branch": cpu_state(content_branch), "content_head": cpu_state(content_head),
            "prosody_branch": cpu_state(pb_final), "recon_head": cpu_state(rh_final), "config": config}, OUT["checkpoint"])
assert os.path.exists(OUT["checkpoint"]) and os.path.getsize(OUT["checkpoint"]) > 0, f"checkpoint was not written to {OUT['checkpoint']}"


def save_training_log():
    """Write every per-epoch row logged so far to training_log_v2.csv."""
    write_csv(pd.DataFrame(log_rows), OUT["log"])


save_training_log()

_ck = torch.load(OUT["checkpoint"], map_location="cpu", weights_only=True)
assert set(_ck) == {"content_branch", "content_head", "prosody_branch", "recon_head", "config"}
_re = build_models()
for _name in ("content_branch", "content_head", "prosody_branch", "recon_head"):
    _re[_name].load_state_dict(_ck[_name])
    _re[_name].to(device).eval()
assert np.allclose(encode(_re["content_branch"], cls_pairs[:20]), C_pairs[:20], atol=1e-5)
assert np.allclose(encode(_re["prosody_branch"], pairs["z_orig"][:20]), D_orig_all[:20], atol=1e-5)
print(f"Saved {OUT['checkpoint']} ({os.path.getsize(OUT['checkpoint']) / 1e6:.2f} MB) and {OUT['log']} ({len(log_rows)} rows); reload check passed")
section_end("F. Save checkpoint and log")

## G. Independence tests on held-out synthetic speakers

In [ ]:
# Independence of C and D on held-out speakers: bias-corrected distance correlation and grouped ridge R2 both ways, each with a 200-shuffle baseline.
section_start("G. Independence tests")
test_groups = pairs["speaker"][idx_test]
C_test = C_pairs[idx_test]
independence = {}
for cond, D_cond in (("original_clips", D_orig_all[idx_test]), ("flattened_clips", D_flat_all[idx_test])):
    independence[cond] = {
        "distance_correlation": distance_correlation_report(C_test, D_cond, N_PERMUTATIONS, SEED),
        "ridge_C_to_D": ridge_report(C_test, D_cond, test_groups, N_PERMUTATIONS, SEED),
        "ridge_D_to_C": ridge_report(D_cond, C_test, test_groups, N_PERMUTATIONS, SEED),
    }
    r = independence[cond]
    print(f"{cond}: dCor2_bc {r['distance_correlation']['dcor2_bias_corrected']:+.4f} (p={r['distance_correlation']['p_value']:.3f}), "
          f"ridge R2 C->D {r['ridge_C_to_D']['r2']:+.3f} (p={r['ridge_C_to_D']['p_value']:.3f}), "
          f"D->C {r['ridge_D_to_C']['r2']:+.3f} (p={r['ridge_D_to_C']['p_value']:.3f})")

D_none_test = encode(pb_none, pairs["z_orig"][idx_test])
ablation = {
    "dcor2_bias_corrected": dcor2_bias_corrected(C_test, D_none_test),
    "ridge_C_to_D_r2": grouped_ridge_r2(C_test, D_none_test, test_groups),
    "ridge_D_to_C_r2": grouped_ridge_r2(D_none_test, C_test, test_groups),
}
tests = [independence[c][k] for c in independence for k in ("distance_correlation", "ridge_C_to_D", "ridge_D_to_C")]
decoupling_verified = all(t["p_value"] >= 0.05 for t in tests)
independence_json = {
    "content_label_source": CONTENT_LABEL_SOURCE,
    "content_scorer_stamp": stamp,
    "data": f"held-out LibriSpeech speakers only: {len(idx_test)} pairs from {len(set(test_groups))} speakers; original and flattened copies tested separately because both share the same C",
    "selected_lambda": float(selected_lambda),
    "independence_penalty": INDEPENDENCE_PENALTY,
    "results": independence,
    "ablation_no_penalty_lambda_0_original_clips": ablation,
    "lambda_sweep": sweep,
    "decoupling_verified": bool(decoupling_verified),
    "decoupling_verified_rule": "True only if all six permutation p-values (dCor, ridge C->D, ridge D->C, for original and flattened clips) are >= 0.05, i.e. none of the tests found C and D related.",
    "plain_language": {
        "dcor2_bias_corrected": "How strongly C and D are related in any way (also non-linear). Bias-corrected: about 0 means unrelated, larger means related; it can be slightly negative by chance.",
        "ridge_r2": "How well a simple linear model, trained on some speakers, predicts D from C (or C from D) on other speakers. Near or below 0 means no linear predictability; clearly above 0 means leakage.",
        "permutation_baseline": "The same statistic recomputed 200 times with the C-D pairing shuffled, which destroys any real link. 'percentile_of_real_in_null' is where the real value falls among those 200 (near 100 = real link; around 50 or below = no link). 'p_value' is the share of shuffles at least as large as the real value.",
        "ablation_no_penalty": "The same statistics for a ProsodyBranch trained with lambda = 0 (reconstruction only). If these are much larger than the selected model's, the independence penalty is what removed the dependence.",
        "decoupling_verified": "A convenience flag from the rule above. Failing to detect a relationship is not proof of independence.",
    },
    "caveats": [
        "Each LibriSpeech speaker reads their own books, so text and speaker identity are linked; the tests can pick up that link.",
        "Held-out set is small (see data); tests have limited power.",
        "Cosine similarity between C and D is not reported as an independence test: they live in different learned spaces.",
    ],
}
write_json(OUT["independence"], independence_json)
print(f"decoupling_verified = {decoupling_verified}; no-penalty ablation dCor2 {ablation['dcor2_bias_corrected']:+.4f}")
section_end("G. Independence tests")

## H. Prosody-only baseline (illustrative comparison model)

In [ ]:
# Baseline: ridge regression from the 62 z-scored features to an arousal proxy; an illustrative comparison model, not an evaluator or vendor tool.
section_start("H. Prosody-only baseline")
import joblib
from sklearn.linear_model import Ridge

comp_idx = [FEATURE_NAMES.index(f) for f in AROUSAL_FEATURES]
fit_idx = np.where(pairs["split"] != "test")[0]
raw_fit = np.concatenate([pairs["gem_orig"][fit_idx], pairs["gem_flat"][fit_idx]]).astype(np.float64)
comp_mean, comp_std = raw_fit[:, comp_idx].mean(axis=0), raw_fit[:, comp_idx].std(axis=0)
comp_std = np.where(comp_std < 1e-8, 1.0, comp_std)


def arousal_target(raw):
    """Hand-built arousal proxy: mean of the z-scored pitch variability, loudness and speaking rate."""
    return ((np.asarray(raw, dtype=np.float64)[:, comp_idx] - comp_mean) / comp_std).mean(axis=1)


Z_base_fit = np.concatenate([pairs["z_orig"][fit_idx], pairs["z_flat"][fit_idx]])
baseline_ridge = Ridge(alpha=1.0).fit(Z_base_fit, arousal_target(raw_fit))
Z_base_test = np.concatenate([pairs["z_orig"][idx_test], pairs["z_flat"][idx_test]])
raw_base_test = np.concatenate([pairs["gem_orig"][idx_test], pairs["gem_flat"][idx_test]])
baseline_r2_test = float(r2_score(arousal_target(raw_base_test), baseline_ridge.predict(Z_base_test)))

baseline_payload = {"ridge": baseline_ridge, "feature_names": FEATURE_NAMES, "arousal_features": AROUSAL_FEATURES,
                    "arousal_component_mean": comp_mean, "arousal_component_std": comp_std, "z_clip": Z_CLIP,
                    "scaler_file": "gemaps_scaler.json", "label": BASELINE_LABEL, "date": RUN_DATE}
if RUN_STAMP:
    baseline_payload["run_stamp"] = RUN_STAMP
joblib.dump(baseline_payload, OUT["baseline"])
print(f"Baseline ({BASELINE_LABEL}): R2 on held-out pairs {baseline_r2_test:.3f}; saved {OUT['baseline']}")
section_end("H. Prosody-only baseline")

## I. Synthetic evaluation on held-out speakers

In [ ]:
# Synthetic evaluation: how much the baseline's score moves between original and flattened copies versus our content score, plus what D keeps.
section_start("I. Synthetic evaluation")
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

base_orig, base_flat = baseline_ridge.predict(pairs["z_orig"][idx_test]), baseline_ridge.predict(pairs["z_flat"][idx_test])
baseline_gaps = np.abs(base_orig - base_flat)
score_orig = encode(content_head, encode(content_branch, cls_pairs[idx_test])).squeeze(-1)
score_flat = encode(content_head, encode(content_branch, cls_pairs[idx_test])).squeeze(-1)
our_gaps = np.abs(score_orig - score_flat)

gap_reduction_pct = 100.0 * (baseline_gaps.mean() - our_gaps.mean()) / (baseline_gaps.mean() + 1e-12)
t_stat, t_p = sps.ttest_rel(baseline_gaps, our_gaps)
recon_orig = r2_variance_weighted(pairs["z_orig"][idx_test], encode(rh_final, D_orig_all[idx_test]))
recon_flat = r2_variance_weighted(pairs["z_flat"][idx_test], encode(rh_final, D_flat_all[idx_test]))

X_sep = np.concatenate([D_orig_all[idx_test], D_flat_all[idx_test]])
y_sep = np.concatenate([np.zeros(len(idx_test)), np.ones(len(idx_test))])
g_sep = np.concatenate([test_groups, test_groups])
oof = np.zeros(len(y_sep))
for tr, te in GroupKFold(n_splits=min(5, len(set(test_groups)))).split(X_sep, y_sep, g_sep):
    oof[te] = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)).fit(X_sep[tr], y_sep[tr]).predict(X_sep[te])
separability_acc = float((oof == y_sep).mean())

synthetic_json = {
    "content_label_source": CONTENT_LABEL_SOURCE,
    "content_scorer_stamp": stamp,
    "data": f"held-out LibriSpeech speakers only: {len(idx_test)} pairs, {len(set(test_groups))} speakers",
    "baseline": BASELINE_LABEL,
    "mean_baseline_gap": float(baseline_gaps.mean()),
    "mean_our_content_gap": float(our_gaps.mean()),
    "gap_reduction_pct": float(gap_reduction_pct),
    "paired_ttest_t": float(t_stat),
    "paired_ttest_p": float(t_p),
    "cohens_d_paired": cohens_d_paired(baseline_gaps, our_gaps),
    "our_content_gap_is_zero_by_construction": True,
    "caveat": "Our content gap is zero by construction: the content score is computed from the transcript only, and an original clip and its flattened copy have the same transcript. The gap reduction, t-test and Cohen's d therefore restate that design property and are NOT evidence that training worked. The informative numbers are in eval_results_independence.json and eval_results_loso.json.",
    "delivery_information_kept_by_D": {
        "reconstruction_r2_original_clips": recon_orig,
        "reconstruction_r2_flattened_clips": recon_flat,
        "original_vs_flattened_classification_accuracy_from_D": separability_acc,
        "plain_language": "Whether D still carries real delivery information: how well the 62 voice features can be rebuilt from D (1 = perfectly), and how often a simple classifier can tell an original clip from its flattened copy using D alone (0.5 = chance).",
    },
    "baseline_r2_on_arousal_proxy_held_out": baseline_r2_test,
}
write_json(OUT["synthetic"], synthetic_json)
print(f"Baseline gap {baseline_gaps.mean():.4f} vs our gap {our_gaps.mean():.4f} (zero by construction); D recon R2 {recon_orig:.3f}/{recon_flat:.3f}; "
      f"original-vs-flattened accuracy from D {separability_acc:.3f}")
section_end("I. Synthetic evaluation")

## J. Real speakers — eight leave-one-speaker-out folds

In [ ]:
# Real clips: transcribe with faster-whisper (same model name as src/api/pipeline.py), embed with frozen RoBERTa, encode with the frozen ContentBranch.
section_start("J. Real speakers (LOSO)")
import gc
from faster_whisper import WhisperModel

roberta.cpu()
torch.cuda.empty_cache()
whisper = WhisperModel(WHISPER_MODEL_NAME, device="cuda", compute_type="float16")
for c in real_clips:
    if not os.path.isfile(c["wav"]) or os.path.getsize(c["wav"]) == 0:
        raise FileNotFoundError(f"Converted clip {c['clip_id']} is missing or empty: {c['wav']}")
    segments, _ = whisper.transcribe(c["wav"], beam_size=3)
    c["transcript"] = " ".join(s.text.strip() for s in segments).strip()
del whisper
gc.collect()
torch.cuda.empty_cache()
roberta.to(device)

write_csv(pd.DataFrame([{"speaker_id": c["speaker_id"], "clip_id": c["clip_id"], "transcript": c["transcript"]} for c in real_clips]), OUT["transcripts"])
valid = np.array([c["transcript"] != "" for c in real_clips])
excluded_empty = {sid: int(sum(1 for c, v in zip(real_clips, valid) if c["speaker_id"] == sid and not v)) for sid in SPEAKER_FOLDER_MAP}
real = {
    "speaker": np.array([c["speaker_id"] for c in real_clips])[valid],
    "clip_id": [c["clip_id"] for c, v in zip(real_clips, valid) if v],
    "z": real_z[valid],
    "zraw": real_zraw[valid],
}
cls_real = embed_texts("real", [c["transcript"] for c, v in zip(real_clips, valid) if v])
real["C"], real["score"] = content_scores(cls_real)
roberta.cpu()
roberta = None
gc.collect()
torch.cuda.empty_cache()
print(f"Real clips: {len(real_clips)} transcribed, {int(valid.sum())} with speech kept; excluded for an empty transcript: {excluded_empty}")

In [ ]:
# LOSO: per held-out speaker, retrain the ProsodyBranch on LibriSpeech-train plus the other 7 speakers (or reuse the main model), then evaluate on the held-out speaker.
no_clips = [sp for sp in SPEAKER_FOLDER_MAP if sp not in set(real["speaker"])]
if no_clips:
    raise RuntimeError(f"No usable clips (non-empty transcript) for {no_clips}; cannot run all {len(SPEAKER_FOLDER_MAP)} folds.")
C_syn_tr, Z_syn_tr = stack_orig_flat(np.where(pairs["split"] != "test")[0])
fold_epochs = max(PROSODY_HP["min_epochs_loso"], best_epoch_final)
loso_mode = ("LOSO with per-fold retraining of the ProsodyBranch on LibriSpeech-train plus the other 7 real speakers" if LOSO_RETRAIN
             else "held-out speaker evaluation, no retraining")
D_oof = np.zeros((len(real["speaker"]), 256), dtype=np.float32)


def fold_independence(C_held, D_held, C_fit, D_fit):
    """Distance correlation plus train-to-held-out ridge R2 in both directions, each with a permutation baseline."""
    return {
        "distance_correlation": distance_correlation_report(C_held, D_held, N_PERMUTATIONS, SEED),
        "ridge_C_to_D_transfer": ridge_transfer_report(C_fit, D_fit, C_held, D_held, N_PERMUTATIONS, SEED),
        "ridge_D_to_C_transfer": ridge_transfer_report(D_fit, C_fit, D_held, C_held, N_PERMUTATIONS, SEED),
    }


loso_folds = []
for fold, spk in enumerate(sorted(set(real["speaker"]))):
    held = real["speaker"] == spk
    others = ~held
    if LOSO_RETRAIN:
        C_fit = np.concatenate([C_syn_tr, real["C"][others]])
        Z_fit = np.concatenate([Z_syn_tr, real["z"][others]])
        pb, rh, _ = train_prosody(C_fit, Z_fit, selected_lambda, fold_epochs, PROSODY_HP, init_seed=SEED + 11, shuffle_seed=SEED + 100 + fold,
                                  run_id=f"loso_{spk}", log=log_rows, val=None)
    else:
        pb, rh = pb_final, rh_final
        C_fit, Z_fit = np.concatenate([C_syn_tr, real["C"][others]]), np.concatenate([Z_syn_tr, real["z"][others]])
    D_held = encode(pb, real["z"][held])
    D_oof[held] = D_held
    n = int(held.sum())
    fold_row = {
        "held_out_speaker": spk,
        "n_clips": n,
        "n_clips_before_exclusion": n + excluded_empty[spk],
        "n_excluded_empty_transcript": excluded_empty[spk],
        "prosody_distance_vs_librispeech": float(np.abs(real["zraw"][held]).mean()),
        "within_speaker_variance": float(real["zraw"][held].var(axis=0).mean()),
        "other_speakers_pooled_variance": float(real["zraw"][others].var(axis=0).mean()),
        "recon_r2": r2_variance_weighted(real["z"][held], encode(rh, D_held)) if n >= 2 else None,
        "mean_content_score": float(real["score"][held].mean()),
    }
    if n >= MIN_CLIPS_FOR_INDEPENDENCE:
        fold_row["independence"] = fold_independence(real["C"][held], D_held, C_fit, encode(pb, Z_fit))
    else:
        fold_row["independence"] = None
        fold_row["independence_skipped_reason"] = f"n < {MIN_CLIPS_FOR_INDEPENDENCE}"
    loso_folds.append(fold_row)
    ind = fold_row["independence"]
    print(f"  {spk}: n={n:3d}, prosody distance {fold_row['prosody_distance_vs_librispeech']:.3f}, within-var {fold_row['within_speaker_variance']:.3f}, "
          + (f"dCor2 {ind['distance_correlation']['dcor2_bias_corrected']:+.4f} (p={ind['distance_correlation']['p_value']:.3f})" if ind else fold_row["independence_skipped_reason"]))
if SMOKE:
    _half = np.arange(len(D_oof)) % 2 == 0
    fold_independence(real["C"][_half], D_oof[_half], real["C"][~_half], D_oof[~_half])
    print("SMOKE: per-fold independence code path executed on pooled rows (not a result)")
save_training_log()

In [ ]:
# LOSO summary: pooled out-of-fold independence, unweighted and clip-weighted aggregates, then save eval_results_loso.json.
speaker_counts = {folder: sum(1 for c in real_clips if c["folder"] == folder) for folder in SPEAKER_FOLDER_MAP.values()}
top2_share = (speaker_counts["S001"] + speaker_counts["S008"]) / sum(speaker_counts.values())

pooled = {
    "n": int(len(D_oof)),
    "distance_correlation": distance_correlation_report(real["C"], D_oof, N_PERMUTATIONS, SEED),
    "ridge_C_to_D": ridge_report(real["C"], D_oof, real["speaker"], N_PERMUTATIONS, SEED),
    "ridge_D_to_C": ridge_report(D_oof, real["C"], real["speaker"], N_PERMUTATIONS, SEED),
    "note": ("Each clip's D comes from the fold model that did not train on its speaker. " +
             ("The 8 fold models share an initial seed but are trained separately, so their D spaces are similar but not identical; this can bias pooled ridge R2 towards 0 and the per-fold transfer results are the cleaner ridge evidence."
              if LOSO_RETRAIN else "All D come from the single main model.")),
}


def aggregate(getter):
    """Unweighted mean/std across folds and clip-weighted mean, skipping folds where the metric is null."""
    vals = [(f["n_clips"], getter(f)) for f in loso_folds if getter(f) is not None and np.isfinite(getter(f))]
    if not vals:
        return None
    w, v = np.array([a for a, _ in vals], dtype=float), np.array([b for _, b in vals], dtype=float)
    return {"n_folds_used": len(vals), "unweighted_mean": float(v.mean()), "unweighted_std": float(v.std(ddof=1)) if len(v) > 1 else None,
            "clip_weighted_mean": float((w * v).sum() / w.sum())}


def _fold_ind(f, key, field):
    return None if f["independence"] is None else f["independence"][key][field]


aggregates = {
    "prosody_distance_vs_librispeech": aggregate(lambda f: f["prosody_distance_vs_librispeech"]),
    "within_speaker_variance": aggregate(lambda f: f["within_speaker_variance"]),
    "recon_r2": aggregate(lambda f: f["recon_r2"]),
    "dcor2_bias_corrected": aggregate(lambda f: _fold_ind(f, "distance_correlation", "dcor2_bias_corrected")),
    "ridge_C_to_D_transfer_r2": aggregate(lambda f: _fold_ind(f, "ridge_C_to_D_transfer", "r2")),
    "ridge_D_to_C_transfer_r2": aggregate(lambda f: _fold_ind(f, "ridge_D_to_C_transfer", "r2")),
}
loso_json = {
    "content_label_source": CONTENT_LABEL_SOURCE,
    "content_scorer_stamp": stamp,
    "loso_mode": loso_mode,
    "loso_retrain": bool(LOSO_RETRAIN),
    "selected_lambda": float(selected_lambda),
    "epochs_per_fold": int(fold_epochs) if LOSO_RETRAIN else None,
    "clips_per_speaker": speaker_counts,
    "n_clips_excluded_empty_transcript_total": int(sum(excluded_empty.values())),
    "clip_imbalance_note": f"S001 and S008 make up {100 * top2_share:.0f} percent of the {sum(speaker_counts.values())} clips and S007 has only {speaker_counts['S007']}; "
                           "folds are very uneven, so the unweighted mean treats each speaker equally while the clip-weighted mean is dominated by S001 and S008.",
    "min_clips_for_independence": MIN_CLIPS_FOR_INDEPENDENCE,
    "folds": loso_folds,
    "aggregate": aggregates,
    "pooled_out_of_fold_independence": pooled,
    "plain_language": {
        "prosody_distance_vs_librispeech": "Average size (in standard deviations) of the speaker's 62 voice features compared with the LibriSpeech scaler: how unusual their delivery looks to the reference population.",
        "within_speaker_variance": "How much the speaker's own clips vary from each other in z-scored delivery features.",
        "recon_r2": "How well D, computed for the held-out speaker, lets the model rebuild that speaker's 62 voice features (1 = perfect).",
        "independence": "Distance correlation and ridge R2 between C and D as in eval_results_independence.json; ridge is fitted on the training clips and scored on the held-out speaker. Null with a reason when the fold has fewer clips than the minimum.",
    },
    "excluded_empty_transcripts": excluded_empty,
    "scope_note": "Eight real speakers only; results are small-sample and descriptive.",
}
write_json(OUT["loso"], loso_json)
print(f"{loso_mode}\nPooled out-of-fold (n={pooled['n']}): dCor2 {pooled['distance_correlation']['dcor2_bias_corrected']:+.4f} "
      f"(p={pooled['distance_correlation']['p_value']:.3f}), ridge C->D R2 {pooled['ridge_C_to_D']['r2']:+.3f}, D->C {pooled['ridge_D_to_C']['r2']:+.3f}")
for k, v in aggregates.items():
    print(f"  {k}: {v}")
section_end("J. Real speakers (LOSO)")

## K. SHAP on the baseline's prosody features

In [ ]:
# SHAP for the prosody-only baseline only; if SHAP fails, fall back to a clearly labelled coefficient proxy and never call it SHAP.
section_start("K. SHAP on the baseline")
attribution = {"explains": "the prosody-only baseline (" + BASELINE_LABEL + "), not the content scorer", "n_explained_rows": int(len(Z_base_test))}
try:
    import shap
    explainer = shap.LinearExplainer(baseline_ridge, Z_base_fit)
    sv = np.asarray(explainer.shap_values(Z_base_test))
    mean_abs = np.abs(sv).mean(axis=0)
    attribution["attribution_method"] = "shap_linear_explainer"
except Exception as e:
    mean_abs = np.abs(baseline_ridge.coef_) * Z_base_fit.std(axis=0)
    attribution["attribution_method"] = "coefficient_magnitude_proxy (NOT SHAP)"
    attribution["shap_error"] = f"{type(e).__name__}: {e}"
order = np.argsort(mean_abs)[::-1][:10]
attribution["top_features"] = [{"feature": FEATURE_NAMES[i], "mean_abs_attribution": float(mean_abs[i])} for i in order]
attribution["mean_abs_attribution_all_features"] = {FEATURE_NAMES[i]: float(mean_abs[i]) for i in range(len(FEATURE_NAMES))}

if not os.path.exists(OUT["synthetic"]):
    raise FileNotFoundError(f"{OUT['synthetic']} was not written by section I; run section I before this cell.")
with open(OUT["synthetic"], "r", encoding="utf-8") as f:
    _syn = json.load(f)
_syn["baseline_attribution"] = to_jsonable(attribution)
write_json(OUT["synthetic"], _syn)
print(f"attribution_method = {attribution['attribution_method']}; top features: {[t['feature'] for t in attribution['top_features'][:5]]}")
section_end("K. SHAP on the baseline")

## L. Figures (from the logs and results above only)

In [ ]:
# Figures: training curves, lambda sweep, independence versus permutation baselines, per-speaker results; saved to /kaggle/working/figures.
section_start("L. Figures")
import matplotlib.pyplot as plt

log_df = pd.DataFrame(log_rows)
subtitle = PROXY_STAMP if stamp else ""


def save_fig(fig, name):
    """Save a figure under FIG_DIR (stamped in smoke mode) and close it."""
    if RUN_STAMP:
        fig.text(0.5, 0.995, RUN_STAMP, ha="center", va="top", color="red", fontsize=12, fontweight="bold")
    fig.savefig(f"{FIG_DIR}/{name}", dpi=150)
    plt.close(fig)


fig, ax = plt.subplots(1, 2, figsize=(11, 4))
c_log = log_df[log_df["stage"] == "content"]
ax[0].plot(c_log["epoch"], c_log["train_mse"], label="train MSE")
ax[0].plot(c_log["epoch"], c_log["es_mse"], label="early-stopping MSE")
ax[0].set_xlabel("epoch"); ax[0].set_ylabel("MSE"); ax[0].set_title("Stage 1: content scorer training"); ax[0].legend()
ax[1].scatter(y_val, pred_val, s=6, alpha=0.4)
ax[1].set_xlabel("ChaLearn interview label"); ax[1].set_ylabel("predicted score")
ax[1].set_title(f"Held-out videos: r = {content_eval['pearson_r_validation']:.2f} {subtitle}")
fig.tight_layout(); save_fig(fig, "fig1_content_scorer.png")

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for lam in LAMBDA_GRID:
    r_log = log_df[(log_df["stage"] == "prosody") & (log_df["run"] == f"stage2_lambda_{lam}")]
    ax[0].plot(r_log["epoch"], r_log["val_recon_mse"], label=f"lambda={lam}")
    ax[1].plot(r_log["epoch"], r_log["val_indep_penalty"], label=f"lambda={lam}")
ax[0].set_title("Validation reconstruction MSE"); ax[1].set_title(f"Validation independence penalty ({INDEPENDENCE_PENALTY})")
for a in ax:
    a.set_xlabel("epoch"); a.legend(fontsize=7)
ax[1].set_yscale("log")
fig.tight_layout(); save_fig(fig, "fig2_prosody_training.png")

panels = [("synthetic original", independence["original_clips"]), ("synthetic flattened", independence["flattened_clips"]),
          ("real speakers, pooled out-of-fold", pooled)]
fig, ax = plt.subplots(1, 3, figsize=(13, 3.8))
for a, (title, rep) in zip(ax, panels):
    d = rep["distance_correlation"]
    a.hist(d["_null_values"], bins=25, color="lightgray", label="200 shuffled pairings")
    a.axvline(d["dcor2_bias_corrected"], color="red", label=f"real (p={d['p_value']:.2f})")
    a.set_title(title, fontsize=9); a.set_xlabel("bias-corrected dCor^2"); a.legend(fontsize=7)
fig.suptitle(f"Distance correlation between C and D {subtitle}", fontsize=10)
fig.tight_layout(); save_fig(fig, "fig3_independence_dcor.png")

fig, ax = plt.subplots(2, 3, figsize=(13, 6))
for j, (title, rep) in enumerate(panels):
    for i, key in enumerate(("ridge_C_to_D", "ridge_D_to_C")):
        r = rep[key]
        ax[i, j].hist(r["_null_values"], bins=25, color="lightgray")
        ax[i, j].axvline(r["r2"], color="red")
        ax[i, j].set_title(f"{title}\n{key.replace('ridge_', 'ridge ').replace('_', ' ')}: R2 {r['r2']:+.2f} (p={r['p_value']:.2f})", fontsize=8)
        ax[i, j].set_xlabel("cross-validated R2")
fig.tight_layout(); save_fig(fig, "fig4_independence_ridge.png")

fig, ax = plt.subplots(1, 4, figsize=(15, 3.8))
names = [f["held_out_speaker"].replace("speaker_", "S") for f in loso_folds]
ax[0].bar(names, [f["n_clips"] for f in loso_folds]); ax[0].set_title("clips per held-out speaker")
ax[1].bar(names, [f["prosody_distance_vs_librispeech"] for f in loso_folds]); ax[1].set_title("prosody distance vs LibriSpeech (SD)")
ax[2].bar(names, [np.nan if f["recon_r2"] is None else f["recon_r2"] for f in loso_folds]); ax[2].set_title("reconstruction R2 (held-out)")
ax[3].bar(names, [np.nan if f["independence"] is None else f["independence"]["distance_correlation"]["dcor2_bias_corrected"] for f in loso_folds])
ax[3].set_title("dCor^2 C-D (blank = n < 10)")
for a in ax:
    a.tick_params(axis="x", labelsize=8)
fig.suptitle(f"Leave-one-speaker-out folds: {loso_mode} {subtitle}", fontsize=9)
fig.tight_layout(); save_fig(fig, "fig5_loso_folds.png")
print(sorted(os.listdir(FIG_DIR)))
section_end("L. Figures")

## M. Final check

In [ ]:
# Final check: every expected output exists and is non-empty, JSON files parse, and the saved checkpoint keys are correct.
section_start("M. Final check")
expected = [OUT[k] for k in ("checkpoint", "scaler", "baseline", "log", "content", "independence", "synthetic", "loso", "transcripts", "gemaps_cache")]
expected += [f"{FIG_DIR}/{n}" for n in ("fig1_content_scorer.png", "fig2_prosody_training.png", "fig3_independence_dcor.png", "fig4_independence_ridge.png", "fig5_loso_folds.png")]
missing = [p for p in expected if not os.path.exists(p) or os.path.getsize(p) == 0]
assert not missing, f"missing or empty outputs: {missing}"
for p in expected:
    if p.endswith(".json"):
        with open(p, encoding="utf-8") as f:
            json.load(f, parse_constant=lambda c: (_ for _ in ()).throw(ValueError(f"{p} contains {c}")))
_final_ck = torch.load(OUT["checkpoint"], map_location="cpu", weights_only=True)
assert set(_final_ck) == {"content_branch", "content_head", "prosody_branch", "recon_head", "config"}
assert all(p.startswith(OUT_DIR) for p in expected), "an expected output path is outside OUT_DIR"
if SMOKE:
    for p in expected:
        if p.endswith(".json"):
            with open(p, encoding="utf-8") as f:
                assert json.load(f).get("run_stamp") == RUN_STAMP, f"{p} is not stamped"
        elif p.endswith(".csv"):
            with open(p, encoding="utf-8") as f:
                assert f.readline().strip() == f"# {RUN_STAMP}", f"{p} is not stamped"
    assert _final_ck["config"].get("run_stamp") == RUN_STAMP and joblib.load(OUT["baseline"]).get("run_stamp") == RUN_STAMP
    print(f"MODE: SMOKE. All smoke outputs present and stamped '{RUN_STAMP}' in {OUT_DIR}. These numbers are not results.")
print("All outputs present and valid:")
for p in expected:
    print(f"  {os.path.getsize(p) / 1024:10.1f} KB  {p}")
print("\ntranscripts_real.csv holds real speakers' speech content: do not commit it to the repository." + ("" if not SMOKE else " (smoke file)"))
section_end("M. Final check")